# RAG 全体を評価して設定を比べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_rag/rag_eval_pipeline.ipynb)

NLP クックブック 22-10 ／ RAG (検索拡張生成)

RAG を「検索できたか」「正しく答えたか」「資料に基づいているか」「答えがないときに断れたか」に分けて測り、設定 (渡す文書の数など) を数値で比べたい。

In [1]:
!pip install -q datasets

In [2]:
import re
import unicodedata
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

# 評価セット (「RAG の評価セットを用意する」と同じ作り方): JSQuAD の段落 + 40 問 + 答えのない質問 5 問
ds = load_dataset("sbintuitions/JSQuAD", split="validation")
passages, pid, seen, eval_set = [], {}, set(), []
for ex in ds:
    c = ex["context"].split(" [SEP] ", 1)[-1]
    if c not in pid:
        pid[c] = len(passages); passages.append(c)
for ex in ds.shuffle(seed=0):
    c = ex["context"].split(" [SEP] ", 1)[-1]
    if pid[c] not in seen and len(eval_set) < 40:
        seen.add(pid[c]); eval_set.append({"q": ex["question"], "answers": sorted(set(ex["answers"]["text"])), "gold": pid[c]})
eval_set += [{"q": q, "answers": [], "gold": None} for q in ["2030 年の冬季オリンピックの開催地はどこですか。", "私の昨日の夕食は何でしたか。",
             "来週の東京の天気はどうなりますか。", "円周率の小数第 100 位の数字は何ですか。", "この文書集合を作った人の名前は何ですか。"]]

embedder = SentenceTransformer("intfloat/multilingual-e5-small")
p_emb = embedder.encode(["passage: " + p for p in passages], normalize_embeddings=True, batch_size=64, show_progress_bar=False)
q_emb = embedder.encode(["query: " + e["q"] for e in eval_set], normalize_embeddings=True)
ranking = np.argsort(-(q_emb @ p_emb.T), axis=1)
print(len(passages), "段落 /", len(eval_set), "問")

1145 段落 / 45 問


In [3]:
name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 3B の日本語モデル (CPU)
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()
nli_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
nli_tok = AutoTokenizer.from_pretrained(nli_name)
nli = AutoModelForSequenceClassification.from_pretrained(nli_name).eval()

@torch.no_grad()
def generate(question, contexts):
    prompt = ("以下の資料だけに基づいて、質問に一文で短く答えてください。資料に答えが書かれていない場合は、推測せずに「わかりません」とだけ答えてください。\n\n"
              "# 資料\n" + "\n".join(f"- {c}" for c in contexts) + f"\n\n# 質問\n{question}")
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    out = llm.generate(**inputs, max_new_tokens=60, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def faithfulness(contexts, answer):
    """回答が、渡した資料のどれかから言えるか (資料ごとの含意確率の最大値)"""
    enc = nli_tok(contexts, [answer] * len(contexts), return_tensors="pt", padding=True, truncation=True, max_length=512)
    return nli(**enc).logits.softmax(-1)[:, 0].max().item()

norm = lambda s: re.sub(r"[\s、。「」『』()（）]", "", unicodedata.normalize("NFKC", s))
REFUSAL = re.compile(r"わかりません|分かりません|記載がありません|書かれていません|情報がありません")

def run(k):
    rows = []
    for i, e in enumerate(eval_set):
        ctx_ids = ranking[i, :k].tolist()
        answer = generate(e["q"], [passages[j] for j in ctx_ids])
        refused = bool(REFUSAL.search(answer))
        rows.append({"q": e["q"], "answerable": e["gold"] is not None, "hit": e["gold"] in ctx_ids if e["gold"] is not None else None,
                     "correct": any(norm(a) in norm(answer) for a in e["answers"]) if e["gold"] is not None else None,
                     "refused": refused, "faithful": None if refused else faithfulness([passages[j] for j in ctx_ids], answer) > 0.5, "answer": answer})
    return pd.DataFrame(rows)

results = {k: run(k) for k in [1, 3]}

In [4]:
def summarize(df):
    a, u = df[df["answerable"]], df[~df["answerable"]]
    answered = df[~df["refused"]]
    return {"検索: 正解段落が入った割合": a["hit"].mean(), "正答率 (答えのある質問)": a["correct"].mean(),
            "正答率 (検索が当たった質問のみ)": a[a["hit"] == True]["correct"].mean(), "誤って断った割合": a["refused"].mean(),
            "忠実性 (答えたもののうち資料で裏付けられた割合)": answered["faithful"].mean(), "正しく断った割合 (答えのない質問)": u["refused"].mean()}

pd.DataFrame({f"k={k}": summarize(df) for k, df in results.items()}).round(3)

,k=1,k=3
検索: 正解段落が入った割合,0.850,0.925
正答率 (答えのある質問),0.750,0.800
正答率 (検索が当たった質問のみ),0.794,0.865
誤って断った割合,0.125,0.050
忠実性 (答えたもののうち資料で裏付けられた割合),0.795,0.750
正しく断った割合 (答えのない質問),0.200,0.600


In [5]:
pd.set_option("display.max_colwidth", 60)
results[3][["q", "hit", "correct", "refused", "faithful", "answer"]].head(8)

,q,hit,correct,refused,faithful,answer
0,出国命令制度を利用できる回数は？,True,True,False,True,出国命令制度を利用できる回数は、生涯で一人、一回限りです。
1,音楽に関する時間に対する構造または組織化を示す時に何と言う語が用いられる傾向がある。,True,True,False,True,音楽に関する時間に対する構造または組織化を示す時に「リズム」という語が用いられる傾向がある。
2,世界第5位の鉱業資源は何？,True,True,False,True,世界第5位のタングステン鉱です。
3,打楽器で音高を持たない物はどんな役割をする？,True,True,False,True,打楽器で音高を持たないものは、主にリズムを奏でる役割を担います。
4,Googleの提供するWebメールアプリケーションは？,True,True,False,True,Googleの提供するWebメールアプリケーションはGmailです。
5,1977年、第3のドメインとして提案されたのは何？,True,True,False,True,1977年、第3のドメインとして提案されたのは古細菌です。
6,江戸落語の「こぶ正」とは誰のことか。,True,True,False,True,江戸落語の「こぶ正」とは林家正吉のことを指します。
7,パンチ法の銅製の母型による活字製造法は誰の発明とされてきたか。,True,True,False,True,パンチ法と銅製の母型による活字製造法は、伝統的にグーテンベルクの発明とされてきた。


- RAG の評価は 4 つに分けて見る: 検索 (正解の文書を渡せたか)、正答 (正しく答えたか)、忠実性 (渡した資料に基づいているか)、棄却 (答えがないときに断れたか)。1 つの数字にまとめると、どこを直せばよいかが分からなくなる。
- 「検索が当たった質問のみの正答率」が低ければ生成側の問題、全体の正答率との差が大きければ検索側の問題。
- 渡す文書を増やすと検索の当たりは増えるが、無関係な文書も増えて、読み違いや遅延の原因になる。両方の数字を見て決める。
- 正答の判定は「正解の文字列を含むか」という簡易なもの。言い換えた正答を誤りと数え、正解の語を含むだけの誤答を正答と数えることがある。この章の「RAG の回答の正しさを LLM に判定させる」と併用する。
- 45 問では、数ポイントの差は誤差の範囲。設定を決めるときは問数を増やし、評価の章のブートストラップで信頼区間を付ける。